# Declarative strategy rules

This notebook separates measured metrics from the rules that turn those metrics into trades. The metrics are analogous to observables; the rule set is the model used to interpret them.

In [ ]:
import numpy as np
import pandas as pd

from cobasket.strategy_rules import (
    MetricCondition,
    StrategyRule,
    StrategyRules,
    compare_rule_strategies,
    run_rule_strategy_backtest,
)

## Synthetic historical inputs

Each metric table has decision dates in rows and tickers in columns. The simulator does not forward-fill these values. A decision is executed on the next price observation.

In [ ]:
rng = np.random.default_rng(12)
dates = pd.date_range('2020-01-01', periods=500, freq='B')
common = 100 + np.cumsum(rng.normal(0.05, 0.6, len(dates)))
prices = pd.DataFrame({
    'AAA': common + rng.normal(0, 1.0, len(dates)),
    'BBB': 0.9 * common + 10 + rng.normal(0, 1.0, len(dates)),
}, index=dates)

evaluation_dates = dates[100::10]
probability = pd.DataFrame({
    'AAA': np.clip(0.5 + rng.normal(0, 0.15, len(evaluation_dates)), 0, 1),
    'BBB': np.clip(0.5 + rng.normal(0, 0.15, len(evaluation_dates)), 0, 1),
}, index=evaluation_dates)
stable = pd.DataFrame(
    rng.random((len(evaluation_dates), 2)) > 0.2,
    index=evaluation_dates,
    columns=prices.columns,
).astype(float)

## Define an ordered strategy

The first matching rule wins. This makes overlapping conditions unambiguous. `is_held` and `current_weight` are supplied automatically by the simulator.

In [ ]:
strategy = StrategyRules(
    name='cointegration plus stability',
    rules=(
        StrategyRule('sell', (MetricCondition('probability', '<=', 0.30),), 0.0),
        StrategyRule(
            'strong buy',
            (
                MetricCondition('probability', '>=', 0.70),
                MetricCondition('stable', '==', True),
            ),
            0.20,
        ),
        StrategyRule(
            'buy',
            (
                MetricCondition('probability', '>=', 0.60),
                MetricCondition('stable', '==', True),
            ),
            0.10,
        ),
        StrategyRule(
            'reduce',
            (
                MetricCondition('probability', '<=', 0.40),
                MetricCondition('is_held', '==', True),
            ),
            0.05,
        ),
    ),
)
strategy.to_dict()

In [ ]:
result = run_rule_strategy_backtest(
    prices,
    {'probability': probability, 'stable': stable},
    strategy,
    initial_cash=10_000.0,
)
pd.Series(result.backtest.metrics)

In [ ]:
result.backtest.equity.plot(title='Rule-strategy equity', ylabel='Portfolio value');

In [ ]:
result.decisions.head(10)

## Compare a simpler strategy

The comparison uses identical prices, decision dates, metrics, capital, and execution timing. This allows us to ask whether an additional condition actually improves held-out performance.

In [ ]:
probability_only = StrategyRules(
    name='probability only',
    rules=(
        StrategyRule('sell', (MetricCondition('probability', '<=', 0.30),), 0.0),
        StrategyRule('buy', (MetricCondition('probability', '>=', 0.60),), 0.10),
    ),
)

summary, detailed = compare_rule_strategies(
    prices,
    {'probability': probability, 'stable': stable},
    (probability_only, strategy),
)
summary

A more complicated rule set should not be selected solely because it performs best on the data used to define it. Compare strategies on validation data and evaluate the final choice on an untouched test period or through nested walk-forward testing.